In [11]:
import sys
from pathlib import Path
from dotenv import dotenv_values
import pandas as pd

# Load environment variables from the .env file
env_vars = dotenv_values()  # Load variables from the .env file

# Get the workspace path from the environment variables
WORKSPACE_PATH = Path(
    env_vars.get("WORKSPACE_PATH", "")
)  # Fetch WORKSPACE_PATH from .env

if not WORKSPACE_PATH:
    raise ValueError("WORKSPACE_PATH is not defined in the .env file or is empty.")

# Add the WORKSPACE_PATH folder to the Python path
sys.path.append(str(WORKSPACE_PATH))

# Import custom utility functions
from src.config.utils import (
    read_csv,
    split_string,
    aggregate_vertical,
    read_excel,
    write_excel,
)

DATA_DIR = Path(env_vars["DATA_DIR"])
RESULTS_DIR = Path(env_vars["RESULTS_DIR"])
MOMENT_OF_SUICIDE_FEATURES = split_string(env_vars["MOMENT_OF_SUICIDE_FEATURES"])
SOCIO_DEMOGRAPHIC_FEATURES = split_string(env_vars["SOCIO_DEMOGRAPHIC_FEATURES"])


In [27]:
def aggregate_sum(df, group_columns=None, value_columns=None, header=None):
    """
    Aggregates a DataFrame using the 'sum' function.
    Creates a hierarchical index for `group_columns` and aggregates numerical `value_columns`.

    Parameters:
    - df (pd.DataFrame): The input DataFrame to aggregate.
    - group_columns (list): List of column names to group by and create hierarchical index.
    - value_columns (list): List of numerical column names to aggregate.
    - header (str or None): Prefix for the resulting aggregation columns. If None, no prefix is added.

    Returns:
    - pd.DataFrame: A DataFrame with hierarchical index and aggregated columns.
    """
    # Validate input
    if group_columns is None or not group_columns:
        raise ValueError("group_columns cannot be None or empty.")
    if value_columns is None or not value_columns:
        raise ValueError("value_columns cannot be None or empty.")

    # Check if value_columns are numeric
    for col in value_columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise ValueError(f"Column '{col}' must be numeric for sum aggregation.")

    # Initialize results
    results = []

    # Process each group column
    for group_col in group_columns:
        # Group by the group column and aggregate the value columns
        grouped = df.groupby(group_col)[value_columns].sum().reset_index()

        # Add "Column" level to the index for hierarchical structure
        grouped["Column"] = group_col
        grouped.set_index(["Column", group_col], inplace=True)

        # Optionally rename the columns with the header prefix
        if header:
            grouped.columns = [f"{header}_{col}" for col in grouped.columns]

        results.append(grouped)

    # Concatenate results for all group_columns
    result_df = pd.concat(results)

    return result_df


In [17]:
data = {
    "Group_A": ["A", "A", "B", "B", "C", "C"],
    "Group_B": ["X", "Y", "X", "Y", "X", "Y"],
    "Year": [2013, 2014, 2013, 2014, 2013, 2014],
    "Value": [10, 20, 30, 40, 50, 60],
    "Other_Value": [5, 15, 25, 35, 45, 55],
}
df = pd.DataFrame(data)


In [28]:
result_sum = aggregate_sum(
    df,
    group_columns=["Group_A", "Group_B"],
    value_columns=["Value", "Other_Value"],
)
result_sum


Value  Other_Value
Column                       
Group_A A     30           20
        B     70           60
        C    110          100
Group_B X     90           75
        Y    120          105

In [26]:
result_sum.columns

Index(['Sum_Value', 'Sum_Other_Value'], dtype='object')